# Example 12.4.1 / 1: Generate Patterns with Named Problems

Companion notebook for **12.4.1 Cutting Stock Example** in the AMPL lab Documentation, section "Generate Patterns with Named Problems". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Generate Patterns with Named Problems

The complete loop below keeps both models in one session as the named problems `Master` and
`Pricing`, as in the Gilmore-Gomory script of the AMPL book. The instance has rolls of width 10
and pieces of width 3, 4 and 5 with demand 5, 4 and 3. The run first asserts that every piece fits
a roll. Options are set after switching to a problem, because each named problem has its own
option environment (Named Problems and Environments (AMPL lab documentation, section 6.5.1)); `presolve` is 0 for the
master because AMPL's presolve returns inconsistent duals for this master, as
Column Generation (AMPL lab documentation, section 12.4) shows. The run switches problems with `ampl.eval("problem
...")`, copies duals into `price` with `get_values("dual")` and item assignment, and appends
`Count` as a new column:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set ITEMS;                                          # item types
param width {ITEMS} > 0;                            # item width
param stock_width > 0;                              # width of one stock roll
param demand {ITEMS} integer >= 0;                  # pieces ordered per item
param nPatterns integer >= 0 default 0;             # columns generated so far
set PATTERNS = 1..nPatterns;                        # current pattern pool
param pieces {ITEMS, PATTERNS} integer >= 0;        # pieces of item i in pattern p
param price {ITEMS} default 0;                      # MeetDemand dual from the master

var UsePattern {PATTERNS} integer >= 0;             # rolls per pattern; relaxed during generation
# pricing: pieces of item i in the new pattern
var Count {i in ITEMS} integer >= 0, <= floor(stock_width / width[i]);

minimize Rolls: sum {p in PATTERNS} UsePattern[p];  # master: stock rolls used
subject to MeetDemand {i in ITEMS}:                 # master: cover the order for each item
    sum {p in PATTERNS} pieces[i,p] * UsePattern[p] >= demand[i];

# pricing: value above 1 = improving column
maximize PatternValue: sum {i in ITEMS} price[i] * Count[i];
# pricing: new pattern fits on one roll
subject to Fits: sum {i in ITEMS} width[i] * Count[i] <= stock_width;

problem Master: UsePattern, Rolls, MeetDemand;      # restricted master problem
problem Pricing: Count, PatternValue, Fits;         # knapsack pricing problem

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["ITEMS"] = ["small", "medium", "large"]
ampl.param["width"] = {"small": 3, "medium": 4, "large": 5}
ampl.param["stock_width"] = 10
ampl.param["demand"] = {"small": 5, "medium": 4, "large": 3}

In [5]:
# run.py
items = ampl.set["ITEMS"].to_list()
width = ampl.param["width"].to_dict()
roll = ampl.param["stock_width"].value()
too_wide = [i for i in items if width[i] > roll]
assert not too_wide, f"pieces wider than the roll: {too_wide}"
patterns = [{j: (int(roll // width[i]) if j == i else 0) for j in items} for i in items]

def load_patterns():
    ampl.param["nPatterns"] = len(patterns)
    ampl.param["pieces"] = {(i, p): pat[i] for p, pat in enumerate(patterns, 1) for i in items}

ampl.eval("problem Master;")
ampl.option["presolve"] = 0
ampl.option["relax_integrality"] = 1
while True:
    load_patterns()
    ampl.eval("problem Master;")
    ampl.solve(solver="highs", verbose=False)
    assert ampl.solve_result == "solved", ampl.solve_result
    ampl.param["price"] = ampl.con["MeetDemand"].get_values("dual").to_dict()
    ampl.eval("problem Pricing;")
    ampl.solve(solver="highs", verbose=False)
    value = ampl.obj["PatternValue"].value()
    print(f"LP rolls {ampl.obj['Rolls'].value():.4g}, best pattern value {value:.4g}")
    if value <= 1 + 1e-6:
        break
    patterns.append({i: round(c) for i, c in ampl.var["Count"].to_dict().items()})

print("patterns:", patterns)
ampl.eval("problem Master;")
ampl.option["relax_integrality"] = 0
ampl.solve(solver="highs", verbose=False)
use = {p: round(v) for p, v in ampl.var["UsePattern"].to_dict().items()}
made = {i: sum(patterns[p - 1][i] * n for p, n in use.items()) for i in items}
used_width = sum(width[i] * made[i] for i in items)
print("integer master:", ampl.solve_result, "rolls", ampl.obj["Rolls"].value(), "use", use)
print("made", made, "surplus", {i: made[i] - ampl.param["demand"][i] for i in items})
print("waste", sum(use.values()) * roll - used_width)

LP rolls 5.167, best pattern value 1.167
LP rolls 4.75, best pattern value 1
patterns: [{'small': 3, 'medium': 0, 'large': 0}, {'small': 0, 'medium': 2, 'large': 0}, {'small': 0, 'medium': 0, 'large': 2}, {'large': 0, 'medium': 1, 'small': 2}]
integer master: solved rolls 6.0 use {1: 2, 2: 2, 3: 2, 4: 0}
made {'small': 6, 'medium': 4, 'large': 4} surplus {'small': 1, 'medium': 0, 'large': 1}
waste 6


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set ITEMS := small medium large;
param width := small 3  medium 4  large 5;
param stock_width := 10;
param demand := small 5  medium 4  large 3;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
problem Master;  option solver highs, solver_msg 0, presolve 0, relax_integrality 1;
problem Pricing; option solver highs, solver_msg 0;
for {i in ITEMS} {
    let nPatterns := nPatterns + 1;
    let {j in ITEMS} pieces[j, nPatterns] :=
        if j = i then floor(stock_width / width[i]) else 0;
}
repeat {
    solve Master;
    let {i in ITEMS} price[i] := MeetDemand[i].dual;
    solve Pricing;
    printf "LP rolls %.4g, best pattern value %.4g\n", Rolls, PatternValue;
    if PatternValue <= 1 + 1e-6 then break;
    let nPatterns := nPatterns + 1;
    let {i in ITEMS} pieces[i, nPatterns] := round(Count[i]);
};
display pieces;
option Master.relax_integrality 0;
solve Master;
display Rolls, UsePattern;
display {i in ITEMS} (sum {p in PATTERNS} pieces[i,p] * UsePattern[p] - demand[i]);
display Rolls * stock_width - sum {i in ITEMS, p in PATTERNS}
    width[i] * pieces[i,p] * UsePattern[p];

LP rolls 5.167, best pattern value 1.167
LP rolls 4.75, best pattern value 1
pieces :=
large  1   0
large  2   0
large  3   2
large  4   0
medium 1   0
medium 2   2
medium 3   0
medium 4   1
small  1   3
small  2   0
small  3   0
small  4   2
;

Rolls = 6

UsePattern [*] :=
1  2
2  2
3  2
4  0
;

sum{p in PATTERNS} pieces[i,p]*UsePattern[p] - demand[i] [*] :=
 large  1
medium  0
 small  1
;

Rolls*stock_width - sum{i in ITEMS, p in PATTERNS} width[i]*pieces[i,p]*
  UsePattern[p] = 6



### Another way in amplpy

The first pricing solve finds the pattern (2, 1, 0) with value 7/6 > 1, which proves an improving
column exists. After it the best value is 1, so the LP bound 4.75 holds for all patterns.
The integer restricted master needs 6 rolls and overproduces one small and one large piece.

A piece wider than the roll stops the run at that assertion, before any solve:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
ampl.set["ITEMS"] = ["small", "medium", "large"]
ampl.param["width"] = {"small": 3, "medium": 4, "large": 11}
ampl.param["stock_width"] = 10
ampl.param["demand"] = {"small": 5, "medium": 4, "large": 3}

In [11]:
# This step fails on purpose, as on the page; the error is printed so the notebook goes on
try:
    # run.py
    items = ampl.set["ITEMS"].to_list()
    width = ampl.param["width"].to_dict()
    roll = ampl.param["stock_width"].value()
    too_wide = [i for i in items if width[i] > roll]
    assert not too_wide, f"pieces wider than the roll: {too_wide}"
    patterns = [{j: (int(roll // width[i]) if j == i else 0) for j in items} for i in items]

    def load_patterns():
        ampl.param["nPatterns"] = len(patterns)
        ampl.param["pieces"] = {(i, p): pat[i] for p, pat in enumerate(patterns, 1) for i in items}

    ampl.eval("problem Master;")
    ampl.option["presolve"] = 0
    ampl.option["relax_integrality"] = 1
    while True:
        load_patterns()
        ampl.eval("problem Master;")
        ampl.solve(solver="highs", verbose=False)
        assert ampl.solve_result == "solved", ampl.solve_result
        ampl.param["price"] = ampl.con["MeetDemand"].get_values("dual").to_dict()
        ampl.eval("problem Pricing;")
        ampl.solve(solver="highs", verbose=False)
        value = ampl.obj["PatternValue"].value()
        print(f"LP rolls {ampl.obj['Rolls'].value():.4g}, best pattern value {value:.4g}")
        if value <= 1 + 1e-6:
            break
        patterns.append({i: round(c) for i, c in ampl.var["Count"].to_dict().items()})

    print("patterns:", patterns)
    ampl.eval("problem Master;")
    ampl.option["relax_integrality"] = 0
    ampl.solve(solver="highs", verbose=False)
    use = {p: round(v) for p, v in ampl.var["UsePattern"].to_dict().items()}
    made = {i: sum(patterns[p - 1][i] * n for p, n in use.items()) for i in items}
    used_width = sum(width[i] * made[i] for i in items)
    print("integer master:", ampl.solve_result, "rolls", ampl.obj["Rolls"].value(), "use", use)
    print("made", made, "surplus", {i: made[i] - ampl.param["demand"][i] for i in items})
    print("waste", sum(use.values()) * roll - used_width)
except Exception as error:
    print(f"{type(error).__name__}: {error}")

AssertionError: pieces wider than the roll: ['large']
